In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here


In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

# Load pretrained EfficientNwtV2-Small
weights = EfficientNet_V2_S_Weights.DEFAULT
model = efficientnet_v2_s(weights=weights)

# Freeze the backbone (feature extractor)
for param in model.features.parameters():
    param.requires_grad = False

# Replace the classifier head
# Get the number of input features for the classifier
in_features = model.classifier[1].in_features
# Replace the last layer with a new one for 26 classes
model.classifier[1] = nn.Linear(in_features, num_classes)

In [ ]:
def train_loop(model, dataloader, optimizer, criterion, device):
    model.train() # Set the model to training mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    for images, labels in dataloader:
        images, labels = images.to(device), labels.to(device)

        # EMNIST letters labels are 1-26, CrossEntropyLoss expects 0-N-1
        labels = labels - 1

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs.data, 1)
        total_samples += labels.size(0)
        correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

def validation_loop(model, dataloader, criterion, device):
    model.eval() # Set the model to evaluation mode
    running_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad(): # Disable gradient calculations during validation
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)

            # EMNIST letters labels are 1-26, CrossEntropyLoss expects 0-N-1
            labels = labels - 1

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    epoch_loss = running_loss / total_samples
    epoch_accuracy = correct_predictions / total_samples
    return epoch_loss, epoch_accuracy

In [ ]:
# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Write your code here


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)),  # TODO: Resize to 28x28
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    transforms.ToTensor(), # TODO: Convert to Tensor
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create DataLoaders
batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train DataLoader batches: {len(train_dataloader)}")
print(f"Test DataLoader batches: {len(test_dataloader)}")

# Display sample images
def imshow(img, title=None):
    # Denormalize
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose((1, 2, 0)) # Convert from (C, H, W) to (H, W, C)
    img = std * img + mean # Denormalize
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    if title is not None:
        plt.title(title)
    plt.axis('off')

# Get a batch of training data
dataiter = iter(train_dataloader)
images, labels = next(dataiter)

# Display images
fig = plt.figure(figsize=(10, 5))
num_images_to_display = 10
for i in range(num_images_to_display):
    ax = fig.add_subplot(2, 5, i + 1, xticks=[], yticks=[])
    imshow(images[i], title=letters[labels[i]-1]) # Adjust label for A-Z indexing
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Define the number of training epochs
num_epochs = 5

# Initialize lists to store metrics
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

# Training loop
print("Starting training...")
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_loop(model, train_dataloader, optimizer, criterion, device)
    val_loss, val_accuracy = validation_loop(model, test_dataloader, criterion, device)

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}:\n"
          f"  Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}\n"
          f"  Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Training complete!")

# Plotting results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), train_accuracies, label='Training Accuracy')
plt.plot(range(1, num_epochs + 1), val_accuracies, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)), # Fixed typo from transrorms
    transforms.Grayscale(3),
    transforms.ToTensor(), # Added as per the overall task description and example solutions
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Fixed missing comma and closing parenthesis
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create DataLoaders
batch_size = 64

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train DataLoader batches: {len(train_dataloader)}")
print(f"Test DataLoader batches: {len(test_dataloader)}")

# Display sample images
def imshow(img, title=None):
    # Denormalize
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img = img.numpy().transpose((1, 2, 0)) # Convert from (C, H, W) to (H, W, C)
    img = std * img + mean # Denormalize
    img = np.clip(img, 0, 1)
    plt.imshow(img)
    if title is not None:
        plt.title(title)
    plt.axis('off')

# Get a batch of training data
dataiter = iter(train_dataloader)
images, labels = next(dataiter)

# Display images
fig = plt.figure(figsize=(10, 5))
num_images_to_display = 10
for i in range(num_images_to_display):
    ax = fig.add_subplot(2, 5, i + 1, xticks=[], yticks=[])
    imshow(images[i], title=letters[labels[i]-1]) # Adjust label for A-Z indexing
plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

# Load pretrained EfficientNwtV2-Small
weights = EfficientNet_V2_S_Weights.DEFAULT
model = efficientnet_v2_s(weights=weights)

# Freeze the backbone (feature extractor)
for param in model.features.parameters():
    param.requires_grad = False

# Replace the classifier head
# Get the number of input features for the classifier
in_features = model.classifier[1].in_features
# Replace the last layer with a new one for 26 classes
model.classifier[1] = nn.Linear(in_features, num_classes)

In [ ]:
print("Model structure after adaptation:")
print(model)

print("\nVerifying feature extractor parameters are frozen:")
for i, param in enumerate(model.features.parameters()):
    if i < 5: # Print for a few parameters to avoid excessive output
        print(f"  Parameter {i+1} requires_grad: {param.requires_grad}")
    if not param.requires_grad:
        pass # All should be False
    else:
        print("  WARNING: Not all feature extractor parameters are frozen!")
        break
print("  (All remaining feature extractor parameters also have requires_grad=False)")

print(f"\nVerifying classifier head output features: {model.classifier[1].out_features}")
if model.classifier[1].out_features == num_classes:
    print(f"  Classifier head correctly set to {num_classes} classes.")
else:
    print("  ERROR: Classifier head output features do not match num_classes!")

In [ ]:
import matplotlib.pyplot as plt

# Set up device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Define the number of training epochs
num_epochs = 5

# Initialize lists to store metrics
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

# Training loop
print("Starting training...")
for epoch in range(num_epochs):
    train_loss, train_accuracy = train_loop(model, train_dataloader, optimizer, criterion, device)
    val_loss, val_accuracy = validation_loop(model, test_dataloader, criterion, device)

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}:\n"
          f"  Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.4f}\n"
          f"  Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}")

print("Training complete!")

# Plotting results
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs + 1), train_losses, label='Training Loss')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs + 1), train_accuracies, label='Training Accuracy')
plt.plot(range(1, num_epochs + 1), val_accuracies, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()